# 12 - Scraping digital data with BeautifulSoup

In this lesson, we will scrape (i.e. extract) digital data using a `Python` library called `BeautifulSoup`. Digital texts have been encoded to be processed by computers, so to scrape or extract digital data from the Internet, we need to understand data encoding.

---
## Lesson Goals
- Basic text encoding (character, structure, Unicode, HTML, XML)
- BeautifulSoup
- Webpage structure
- Using Python dictionaries for structured data extracted from the Internet
- Using Pandas dataframe to store and visualise structured data

**Key Concepts** bytes, Unicode, HTML, XML, BeautifulSoup, dict, pd.DataFrame

## Text encoding

### _Character encoding_

Computers store texts as a stream of numbers called **bytes**. A **character encoding** is a table that says which byte patterns stand for which characters. The universal character encoding standard **Unicode** is the most common and it represents text and symbols from all writing systems around the world.

The Unicode Transformation Format, **UTF**, is a method to encode unicode characters for storage and communication. The different formats specify how Unicode characters will be converted into sequences of bytes. UTF-8, UTF-16, UTF-32 are the most common UTF.

For example, in UTF-8, the sequence `0x43 0x61 0x66 0xE9` is read as “Café”. Modern computer languages use the UTF-8 character set as default.

The UTF-8 table gives the correspondence between characters and codes: https://www.w3schools.com/charsets/default.asp

### _Structure encoding_

Besides the characters, the structure of texts can also be encoded. The encoding of a text's structure is written in a **markup** language that encodes the titles, paragraphs, and other structural elements. Correct byte encoding ensures the right letters appear; correct markup gives those letters a clear structure that browsers and analytical tools can interpret.

#### HTML

> The most famous markup language is probably `HTML` for _HyperText Markup Language_. It was designed in the early 1990s to describe the *presentation* of hyper-documents, in other words, how text, images, and links should appear in a browser. Because its original goal was speed and tolerance for human error, HTML is permissive: tags can be omitted or improperly nested, and most browsers will “guess” what was meant. That tolerance, however, also makes raw HTML notoriously *messy* for scholars: the same structural element (say, a chapter title) might be marked with an `<h2>`, a bold `<p>`, a styled `<div>`, or nothing at all. See the introduction to `HTML` by the w3schools: https://www.w3schools.com/html/html_intro.asp.

#### XML

> The other most common markup language is `XML`, for _eXtensible Markup Language_, which was introduced by the W3C in 1998. `XML` takes the opposite approach from `HTML`: it is a strict, self-describing meta-language for creating custom markup vocabularies (e.g., TEI for literary texts, JATS for journal articles). Every tag must open and close properly, elements cannot overlap, and documents must conform to a schema (such as DTD/XSD). That rigidity is a feature, not a bug: it guarantees that software and scholars can rely on explicit, predictable structure across large corpora. See the introduction to `XML` by w3schools: https://www.w3schools.com/xml/xml_whatis.asp.

You can learn more about `HTML` and `XML` in https://www.geeksforgeeks.org/html/html-vs-xml/

In our notebook 01, we have seen an instance of an XML document. The file with the matadata for the first book from the Project Gutenberg, called `pg1.rdf`, is written in XML. If you open it again, you should now be able to understand the first line of the text:

`<?xml version="1.0" encoding="utf-8"?>`

In the Project Gutenberg, texts are encoded in `UTF-8` and they are available in both `HTML` and `XML` formats.

### PG's text and link structures

In the Project Gutenberg corpus, texts have a unique ID. Each link to a text and its metadata follows a strict pattern:

The metadata are stored under this type of link:

`https://www.gutenberg.org/` + `ebooks/` + `ID`

And the `HTML` version of a text is stored under this type of link:

`https://www.gutenberg.org/` + `cache/epub/` + `ID/pgID` + `-images.html`

You can see the `HTML` version of any books in PG by adding in front of the link `view-source`:

view-source:https://www.gutenberg.org/cache/epub/53450/pg53450-images.html

Now, if you remove **view-source:** from the link, you can see that the browser **interprets** the `HTML` tags and renders the text in a much more readable fashion.

https://www.gutenberg.org/cache/epub/53450/pg53450-images.html

In this lesson, we will explore a Python library called `BeautifulSoup`, which allows us to scrape or extract data from the Internet while preserving the data structure.

### BeautifulSoup

`BeautifulSoup` is a Python library that turns raw `HTML` or `XML` into a *parse tree*, that is a navigable representation of every structural tag, attribute, and piece of text in the document. Once you have that tree, you can search it with `BeautifulSoup` own **methods**, such as `find()` and `find_all()`, navigate through the text's sections (parent ↔ child ↔ sibling), and edit or export exactly the parts you need.

Source: https://www.crummy.com/software/BeautifulSoup/bs4/doc/


**Why we need BeautifulSoup**

| Typical scraping pain-point                                                                | How BeautifulSoup helps                                                                              |
| ------------------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------- |
| Real-world pages are “messy”—missing closing tags, nested tables, inline styles.           | The underlying *parser* (“lxml”, “html5lib”, etc.) cleans up broken markup before you ever touch it. |
| You often want *structure* (e.g., all chapter headings) rather than arbitrary strings.     | Search by tag name, class, id, attributes, or CSS path—no fragile regexes.                           |
| Downstream DH methods (tokenisation, NER, topic modelling) need plain, de-tagged text.     | Call `.get_text()` once you’ve isolated the element(s) you want; everything else is discarded.       |
| You sometimes need to *rewrite* the HTML (remove `<small>` editorial notes, for instance). | Each node is mutable: `tag.decompose()`, `new_tag()`, `unwrap()`, etc.              |

**Key takeaways**

* BeautifulSoup is the *bridge* between the web’s messy markup and Python’s text-analysis ecosystem.
* You write *declarative* queries (“give me every `<i>` tag inside Chapter III”) rather than brittle pattern-matching.
* Once the text is parsed, you’re free to focus on humanities questions—*not* the quirks of tag soup.

With just a few lines, we can go from any text to a tidy corpus ready for all sorts of advanced computational analyses.

The Programming Historian has a lesson on BeautifulSoup, which you can consult for further information:

https://programminghistorian.org/en/lessons/retired/intro-to-beautiful-soup

### BeautifulSoup in practice

We start with the usual import.

In [ ]:
# @title Grant GoogleColab access to your GoogleDrive and import questions for this notebook
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add your module folder to Python path
import sys
module_path = f"/content/drive/My Drive/IDH/Notebooks"
sys.path.append(module_path)
print("GoogleColab can now access your GoogleDrive.")

# Reload and re-import when editing the module
# Uncomment if the file has been edited
# Import your module
import importlib
import QuestionsBeautifulSoup
importlib.reload(QuestionsBeautifulSoup)

# Import exercises
from QuestionsBeautifulSoup import E1, E2, E3, E4, E5, E6, E7, question, solution

We also need to import specific libraries.

In [ ]:
# BeautifulSoup parses HTML/XML;
# SoupStrainer limits parsing to selected tags/sections for speed.
# requests fetches a web page: resp = requests.get(url, timeout=10); html = resp.text

from bs4 import BeautifulSoup, SoupStrainer
import requests
# Import pandas to build dataframes and time
import pandas as pd, time

Let's say we want to work on onld medical texts to understand how medical writings have evolved throughout the 18th and 19th centuries. We start with an example: _The London Medical Gazette_ from December 27, 1828, with **ID=53450**, which we have just seen above. We will use the RDF file that we have already explored.

In [ ]:
# Set text ID
TEXT_ID = 53450
# Request the metadata for 53450
resp = requests.get(f"https://www.gutenberg.org/ebooks/{TEXT_ID}.rdf", timeout=15)
# Store the request's response in a variable
soup = BeautifulSoup(resp.content, "xml")

We have now the metadata of _The London Medical Gazette_ stored in a variable called `soup`.

`soup` is a special object. Use the next cell to print the type of `soup`.

### Exercise 1

In [ ]:
# Write your line of code to print the type of `soup`


In [ ]:
# Run this cell if you are not sure about your answer.
solution(E1)

If you run the next cell, it will print the content of the `soup` and you will see again this RTF file content, which is so difficult to read for humans.

In [ ]:
# Print the content of the coup to see how it looks like
print(soup.prettify())

---------------

_**Find specific elements within the `soup`**_

With BeautifulSoup, we can select the information we need, providing we know the structure of our file. For example, we can get the tile using the `.find()` method:

In [ ]:
print(soup.find("dcterms:title"))

------------------
_**Get only the text**_


We can also remove the tags and get only the text:

In [ ]:
print(soup.find("dcterms:title").get_text(strip=True))

### Exercise 2

Following the example of the title, how would you print the **subject** of the text 53450?

In [ ]:
question(E2)

--------------------------------------------------
To organise our metadata, we can build a `dictionary`.

The `keys` of the dictionary will be

- Title,
- Author,
- Language,
- Subject.

And the `values` of the dictionary will the extracted text.

In [ ]:
bib = {
    "Title":   soup.find("dcterms:title").get_text(strip=True),
    "Author":  soup.find("pgterms:name").get_text(strip=True),
    "Language": soup.find("dcterms:language").find("rdf:value").get_text(strip=True),
    "Subjects": soup.find("dcterms:subject").find("rdf:value").get_text(strip=True)
}

In [ ]:
bib

### Exercise 3

We can get the title using the key 'Title'. Write the line of code that will print the title using the dictionary that we just built.

In [ ]:
question(E3)

To build a `dictionary` for a single title does not make sense, since it contains the same data as the `soup` we have already built. Run the next cell and you will see that if the two statements are equal, you will get a `True`.

In [ ]:
bib['Title'] == soup.find("dcterms:title").get_text(strip=True)

The dictionary becomes useful when we want to store various titles, or `soups`, that we have scraped. Instead of having `soup_1`, `soup_2`, etc. we store the `soups` iteratively in a dictionary. In the next cells we will build a dictionary with the data from all the titles in the Project Gutenberg corpus that relate to medicine and were written before the 18th century. The subject page is the following:

https://www.gutenberg.org/ebooks/subject/8129

As you can see, the subject has an ID: **8129**.

We can use this link to build a new `soup` with the data from the titles that match this subject.

In [ ]:
# Store the URL into a variable
url = "https://www.gutenberg.org/ebooks/subject/8129"

# Request the URL content using REQUESTS
resp = requests.get(url, timeout=15)

# Build the soup
soup = BeautifulSoup(resp.text, "html.parser")

We can now create a list of dictionaries that we call `records` to store the data from each title using a `loop`. If you print the soup, you will see that the information related to each title is stored between two specific tags. The opening tag is:

`<li class="booklink">`

And the closing tag is:

`</li>`

With our `soup`, we can easily select this information. For example, if we want the booklink information for the first title, we can use our soup as follows:

In [ ]:
# The first title is the title 0 in Python
soup.select("li.booklink")[0]

-------------------
We can also get the title if we use the methods `.select_one()` and `.get_text()`  after the selection of the first title :

In [ ]:
soup.select("li.booklink")[0].select_one(".title").get_text()

### Exercise 4

Following the example above and looking at the `<li class="booklink">` content, can you guess how to get the number of downloads for the last title? Remember that Python starts counting with 0.

In [ ]:
question(E4)

---

If you have one title, it is not a problem to write such a long line of code. But if you have many titles, it becomes difficult. This is where our dictionaries and dataframes are usefule. Let's loop of our `soup` and store each title and its meatadata in a list of dictionaries that we will turn into a dataframe.

In [ ]:
# We start by creating an empty list
records = []

# We loop over each booklink in our soup
for link in soup.select("li.booklink"):
    title = link.select_one(".title") # We store the title in the variable "title"
    subtitle = link.select_one(".subtitle") # The author is stored under the tag "subtitle"
    downloads = link.select_one(".extra") # The tag extra contains the number of downloads
    link = link.select_one("a[href^='/ebooks/']") # We keep the URL for each title
    # Within the loop we create a dictionary
    # for each title and its information
    # and we store each dictionary in the records list we created initially
    records.append({
        "Title": title.get_text(strip=True) if title else None,
        "Author": subtitle.get_text(strip=True) if subtitle else None,
        "Downloads": downloads.get_text(strip=True) if downloads else None,
        "URL": f"https://www.gutenberg.org{link['href']}" if link else None
    })

Run the next cell and you will see that the information for each title is now nicely stored in a tidy list of dictionaries.

In [ ]:
records

----------------------------
We can further improve the visualisation of our data by creating a dataframe.

### Exercise 5

Create a dataframe called `pgmed1800` using `pd.DataFrame()` and the variable in which we have stored the list of dictionaries created above.

In [ ]:
# Write your line of code below


In [ ]:
# Run this cell to see the results
pgmed1800

In [ ]:
# Run this cell if you do not obtain the right DataFrame
solution(E5)

----------------
With the dataframe, we have **easy access** to the different types of data extracted from PG's page. For example, we can get a list of the authors of medical texts prior to 1800:

In [ ]:
# We build a list and drop the duplicate names with `.unique()`
authors = pgmed1800['Author'].unique().tolist()

We can further use this structure to obtain more information. For example, we can extract Wikipedia information about these authors by using their names. Like the Project Gutenberg, Wikipedia uses a clear pattern to build its hyperlinks. The link to a person's page in English will usually be:

`https://en.wikipedia.org/wiki/` + `FIRST_LAST`

Hence the link to our first author, Erasmus Darwin, is:

https://en.wikipedia.org/wiki/Erasmus_Darwin

We can again build a `soup` to scrape the information about Erasmus Darwin. If you look at the Wikipedia page for Erasmus Darwin, you will see that his biographical information is stored in a box. This box is called `infobox` and we can obtain its content by looking for the `table` of the `"class": "infobox"`.

In [ ]:
# Store the URL in a variable
url = "https://en.wikipedia.org/wiki/Erasmus_Darwin"

# Send a descriptive User-Agent (and optionally Accept-Language)
headers = {
    "User-Agent": "Introduction to Digital Humanities (contact: mbednarkiewicz@faculty.ie.edu)",
    "Accept-Language": "en",
}

# Request the page content
resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()  # fail fast on 4xx/5xx
# Store the request's response in a soup
soup = BeautifulSoup(resp.text, "html.parser")

# Use a CSS selector; it matches even if the class list is "infobox biography vcard"
infobox = soup.select_one("table.infobox")

print("Title:", soup.title.get_text(strip=True))
print("Found infobox?", bool(infobox))

In [ ]:
print(infobox.get_text(" ", strip=True))

---------------

To get this information in a line is not particularly useful. Thus we can use a dictionary that we can then easily turn into a dataframe.

### Exercise 6

Use the next cell to create an empty dictionary called `data`. Be careful to use the right brackets.

In [ ]:
# Create an empty dictionary


In [ ]:
# Check if you created a dictionary
type(data) == dict

In [ ]:
solution(E6)

We can now use your dictionary to store our data in a structured way.

In [ ]:
# Loop over the lines in our infobox
for row in infobox.select("tr"):
    header = row.find("th") # Build the keys of the dictionary with the infobox headers
    cell = row.find("td") # Build the values of the dictionary with the infobox content
    if header and cell:
        data[header.get_text(" ", strip=True)] = cell.get_text(" ", strip=True)
print(data)

In [ ]:
# Build a dataframe to improve the visualisation of the data
df = pd.DataFrame(list(data.items()), columns=["Header", "Value"])
df

----------------------
Now that we have the code for one author, we can try to generate a table for all the authors in our small sub-corpus of early medical texts up to 1800. We will use the following steps:

1. Create an empty list to store the URLs from Wikipedia.
2. Generate the Wikipedia links by coupling the FIRST and LAST names of each author.
3. Create an empty list to store the dictionaries with the authors' information from Wikipedia's infoboxes.
4. Create an empty list to store the potential errors that we might encounter.
5. Reuse the code we wrote above to scrape the infoboxes from each author if available.
6. Create a dataframe to visualise our data.

In [ ]:
# 1. Create an empty list to store the URLs from Wikipedia.
wiki_url = []

In [ ]:
# 2. Generate the Wikipedia links by coupling the FIRST and LAST names of each author.
for name in authors:
    wiki_url.append(f"https://en.wikipedia.org/wiki/{name.replace(" ", "_")}")

In [ ]:
# Check if you have the right links
wiki_url

In [ ]:
# 3. Create an empty list to store the dictionaries with the authors' information from Wikipedia's infoboxes.
authors_data = []
# 4. Create an empty list to store the potential errors that we might encounter.
errors = []

In [ ]:
# 5. Scrape the infoboxes for each author if available.

# Send a descriptive User-Agent (and optionally Accept-Language)
headers = {
    "User-Agent": "Introduction to Digital Humanities (contact: mbednarkiewicz@faculty.ie.edu)",
    "Accept-Language": "en",
}

# Loop over the list of Wikipedia URLs
for link in wiki_url:
    # Request the URLs content
    resp = requests.get(link, headers=headers, timeout=15)
    # Store the requests' response in a soup
    soup = BeautifulSoup(resp.text, "html.parser")
    # Select the infobox content
    infobox = soup.find("table", {"class": "infobox"})
    # Create an empty dictionary to store the infobox content for each author
    data = {}
    # Use `try` to handle potential errors and allow the code to run regardless
    try:
        # Loop of the infobox content
        for row in infobox.select("tr"):
            header = row.find("th") # Build the keys of the dictionary with the infobox headers
            cell = row.find("td") # Build the values of the dictionary with the infobox content
            if header and cell:
                data[header.get_text(" ", strip=True)] = cell.get_text(" ", strip=True)
        authors_data.append(data)
    # Track potential errors to spot the missing information
    except Exception as e:
        errors.append({"WikipediaURL": link, "Error": str(e)})
    time.sleep(1)


In [ ]:
# Check the list of dictionaries looks correct
authors_data

In [ ]:
# Check if there were errors
errors

We have an error with John Hill, for whom the Wikipedia page does not contain an infobox. The infobox we create for John Hill is therefore a `NoneType` object. If we go to the Wikipedia page, we see that there are many John Hill and the link we actually need is:

https://en.wikipedia.org/wiki/John_Hill_(botanist)

The parenthese have been added to disambiguate the name and distinguish our John Hill from all the others. When working with large dataset, error handling is key and it is crucial to think about all the errors that can occur. Researchers must then be creative to find workable solutions. In this case we would need to add John Hill's link manually for instance, and since we have only one error, it is feasible. The larger the dataset is, the higher the risk of errors become, and the more creative we have to be to find solutions.

If we build a dataframe, we will have our data for three authors out of four, because John Hill has been excluded.

### Exercise 7

Build the DataFrame as you did in the previous exercise and call it `pgmed1800_authors`.

In [ ]:
# Build the DataFrame


In [ ]:
# Check that your code has worked
pgmed1800_authors

In [ ]:
# Run the next cell if you did not get the right answer.
solution(E7)

## Lesson Summary

- Digital text characters are usually encoded following the **UTF** scheme.
- Digital text structure is usually encoded following variants of `HTML` or `XML` schemes.
- We use BeautifulSoup to navigate large number of digital texts and extract information from them.
- Structured data can be visually easier to analyse in a Python Pandas DataFrame.